In [ ]:
# ── CÉLULA DE SETUP  (rode esta célula para iniciar) ─────────────────────────────────
import os
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

from google.colab import files
print("Selecione o arquivo desafio_nps_fase_1.csv")
uploaded = files.upload()

for nome, conteudo in uploaded.items():
    with open(nome, "wb") as f:
        f.write(conteudo)
print("CSV carregado com sucesso!")


## 02. Feature Engineering e Pré-processamento

Esse notebook tem como objetivo a preparação dos dados para modelagem, remoção de colunas com data leakage, tratamento de variáveis categóricas, criação novas features e salvar o dataset processado para usar no notebook de modelagem.

---


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

print("Bibliotecas carregadas!")


## 1. Carregamento dos Dados


In [ ]:
df = pd.read_csv("desafio_nps_fase_1.csv")
print(f"Shape: {df.shape}")
df.head()


## 2. Criação da Variável Target

Criamos a coluna `nps_category` (Detrator/Neutro/Promotor) que será usada como target nos modelos de classificação.


In [ ]:
def classificar_nps(nota):
    if nota <= 6:
        return "Detrator"
    elif nota <= 8:
        return "Neutro"
    else:
        return "Promotor"

df["nps_category"] = df["nps_score"].apply(classificar_nps)

print("Distribuição das categorias NPS:")
print(df["nps_category"].value_counts())
print()
print((df["nps_category"].value_counts(normalize=True)*100).round(1))


## 3. Remoção de Colunas com Data Leakage

| Coluna | Motivo |
|---|---|
| `csat_internal_score` | Coletado simultaneamente ao NPS |
| `repeat_purchase_30d` | Evento que ocorre 30 dias **após** a compra |
| `customer_id` | Identificador sem valor preditivo |
| `order_id` | Identificador sem valor preditivo |


In [ ]:
# Separando os targets ANTES de remover as colunas
target_continuo = df["nps_score"].copy()       # para regressão
target_categorico = df["nps_category"].copy()  # para classificação

# Removendo colunas com leakage e IDs
colunas_remover = ["csat_internal_score", "repeat_purchase_30d",
                   "customer_id", "order_id", "nps_score", "nps_category"]

df_features = df.drop(columns=colunas_remover, errors="ignore")

print("Colunas removidas:", colunas_remover)
print(f"Shape após remoção: {df_features.shape}")
print("\nColunas restantes:")
print(df_features.columns.tolist())


## 4. Encoding de Variáveis Categóricas

A coluna `customer_region` é do tipo texto, mas modelos de Machine Learning não aceitam texto diretamente. Sendo assim, aplicamos **One-Hot Encoding** (de: variável para: numeral) onde cada região vira uma coluna binária (0 ou 1).


In [ ]:
df_features = pd.get_dummies(df_features, columns=["customer_region"], drop_first=False)

print(f"Shape após encoding: {df_features.shape}")
print("\nColunas de região geradas:")
print([c for c in df_features.columns if "customer_region" in c])


## 5. Feature Engineering (Criação de Novas Variáveis)

Criamos novas features combinando variáveis existentes para capturar padrões relevantes:

| Feature | Cálculo | Intuição de negócio |
|---|---|---|
| `discount_ratio` | desconto/valor do pedido | % de desconto recebido |
| `has_delay` | 1 se houve atraso, 0 se não | flag binária de atraso |
| `high_contact` | 1 se contatos ≥ 3 | cliente que precisou insistir muito |
| `freight_per_item` | frete/qtde de itens | custo de frete por item |


In [ ]:
df_features["discount_ratio"]   = df_features["discount_value"] / (df_features["order_value"] + 1e-9)
df_features["has_delay"]        = (df_features["delivery_delay_days"] > 0).astype(int)
df_features["high_contact"]     = (df_features["customer_service_contacts"] >= 3).astype(int)
df_features["freight_per_item"] = df_features["freight_value"] / (df_features["items_quantity"] + 1e-9)

print("Novas features criadas:")
print(df_features[["discount_ratio", "has_delay", "high_contact", "freight_per_item"]].describe().round(3))


## 6. Salvando o Dataset Processado

Adicionamos os targets de volta ao dataset e salvamos o arquivo processado.


In [ ]:
df_final = df_features.copy()
df_final["nps_score"]    = target_continuo.values
df_final["nps_category"] = target_categorico.values

df_final.to_csv("nps_processed.csv", index=False)

print(f"Dataset processado salvo!")
print(f"Shape final: {df_final.shape}")
print(f"\nColunas finais:")
print(df_final.columns.tolist())
df_final.head()


In [ ]:
# ── CÉLULA FINAL — baixa o dataset processado ────────────────────────────
# IMPORTANTE: faça o download desse arquivo e faça upload no notebook 03
from google.colab import files
files.download("nps_processed.csv")
print("Download iniciado! Salve o arquivo nps_processed.csv.")
